In [1]:
import re
import numpy as np

In [2]:
def get_query_tokens(query):
    """
    Mengambil token/kata dari query.
    Contoh:
    'kopi arabika toraja' -> ['kopi', 'arabika', 'toraja']
    """
    return re.findall(r"\w+", str(query).lower())

In [3]:
def relevant_mask(series, query, threshold=0.5):
    """
    Menentukan apakah item dianggap relevan terhadap query.

    Item dianggap relevan jika proporsi token query yang muncul
    pada nama produk >= threshold.

    Contoh:
    query = 'kopi arabika toraja'
    name_clean = 'kopi toraja asli'

    token query = ['kopi', 'arabika', 'toraja']
    token yang cocok = ['kopi', 'toraja'] = 2 token

    relevance score = 2/3 = 0.67

    Jika threshold = 0.5, maka produk dianggap relevan.
    """

    q_tokens = get_query_tokens(query)

    if len(q_tokens) == 0:
        return series.apply(lambda x: False)

    def score(text):
        text_tokens = set(re.findall(r"\w+", str(text).lower()))
        match_count = sum(token in text_tokens for token in q_tokens)
        return match_count / len(q_tokens)

    return series.apply(score) >= threshold

In [4]:
def precision_at_k(df_ranked, query, k, threshold=0.5, text_col="name_clean"):
    """
    Precision@K:
    Mengukur proporsi produk relevan dalam K hasil teratas.

    Rumus:
    Precision@K = jumlah produk relevan pada top-K / K
    """

    if df_ranked.empty or k <= 0:
        return 0.0

    df_k = df_ranked.head(k)

    if len(df_k) == 0:
        return 0.0

    rel_mask = relevant_mask(df_k[text_col], query, threshold)

    return float(rel_mask.sum() / len(df_k))

In [5]:
def recall_at_k(df_ranked, df_all, query, k, threshold=0.5, text_col="name_clean"):
    """
    Recall@K:
    Mengukur seberapa banyak produk relevan yang berhasil ditemukan
    pada top-K dibandingkan seluruh produk relevan yang ada pada candidate pool.

    Rumus:
    Recall@K = jumlah produk relevan pada top-K / jumlah seluruh produk relevan
    """

    if df_ranked.empty or df_all.empty or k <= 0:
        return 0.0

    all_rel_mask = relevant_mask(df_all[text_col], query, threshold)
    total_relevant = int(all_rel_mask.sum())

    if total_relevant == 0:
        return 0.0

    df_k = df_ranked.head(k)
    topk_rel_mask = relevant_mask(df_k[text_col], query, threshold)

    return float(topk_rel_mask.sum() / total_relevant)

In [6]:
def f1_at_k(precision, recall):
    """
    F1@K:
    Menggabungkan Precision@K dan Recall@K menjadi satu nilai harmonik.

    Rumus:
    F1@K = 2 * Precision * Recall / (Precision + Recall)
    """

    if precision + recall == 0:
        return 0.0

    return float(2 * precision * recall / (precision + recall))

In [7]:
def ndcg_at_k(df_ranked, query, k, threshold=0.5, text_col="name_clean"):
    """
    NDCG@K:
    Mengukur kualitas urutan ranking.

    Pada versi ini, relevance gain dibuat biner:
    - 1 jika produk relevan terhadap query
    - 0 jika tidak relevan

    Ini lebih aman daripada memakai bm25_score langsung sebagai relevance,
    karena BM25 adalah skor ranking sistem, bukan ground truth relevansi.
    """

    if df_ranked.empty or k <= 0:
        return 0.0

    df_k = df_ranked.head(k).copy()

    if len(df_k) == 0:
        return 0.0

    rel = relevant_mask(df_k[text_col], query, threshold).astype(int).to_numpy()

    discounts = 1 / np.log2(np.arange(2, len(rel) + 2))
    dcg = np.sum(rel * discounts)

    ideal_rel = np.sort(rel)[::-1]
    idcg = np.sum(ideal_rel * discounts)

    if idcg == 0:
        return 0.0

    return float(dcg / idcg)

In [8]:
def evaluate_at_k(df_ranked, df_all, query, k, threshold=0.5, text_col="name_clean"):
    """
    Menghitung semua metrik evaluasi untuk satu query pada nilai K tertentu.
    """

    precision = precision_at_k(
        df_ranked=df_ranked,
        query=query,
        k=k,
        threshold=threshold,
        text_col=text_col
    )

    recall = recall_at_k(
        df_ranked=df_ranked,
        df_all=df_all,
        query=query,
        k=k,
        threshold=threshold,
        text_col=text_col
    )

    f1 = f1_at_k(precision, recall)

    ndcg = ndcg_at_k(
        df_ranked=df_ranked,
        query=query,
        k=k,
        threshold=threshold,
        text_col=text_col
    )

    return {
        f"Precision@{k}": precision,
        f"Recall@{k}": recall,
        f"F1@{k}": f1,
        f"NDCG@{k}": ndcg
    }

In [ ]:
query = "oleh-oleh daerah"
k = [20, 30, 40, 50, 60, 70, 80, 90, 100]

# df_ranked = hasil ranking sistem, misalnya hasil BM25 atau Hybrid
# df_all = seluruh candidate pool yang menjadi pembanding recall

result = evaluate_at_k(
    df_ranked=df_hybrid_result,
    df_all=bm25_candidates,
    query=query,
    k=k,
    threshold=0.5,
    text_col="name_clean"
)

print(result)